# 01. 기초: Pretraining-SFT-RL 파이프라인

목표: 논문의 체스 테스트베드를 작은 토큰 시퀀스로 흉내 내고, 사전학습·SFT·RL 단계가 각각 무엇을 학습하는지 이해합니다.

실행 방법: 위에서부터 셀을 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 체스 수를 토큰으로 직렬화하기

논문은 체스 수를 기물, 출발 칸, 도착 칸, 플래그 토큰으로 표현합니다. 여기서는 아주 작은 예시만 만듭니다.

In [ ]:
def serialize_move(piece, source, destination, flag="-"):
    # 모델 입장에서는 체스판이 아니라 토큰 시퀀스를 다음 토큰 예측 대상으로 봅니다.
    return [f"<{piece}>", f"<{source}>", f"<{destination}>", f"<{flag}>"]


game = []
game += serialize_move("P", "e2", "e4")
game += serialize_move("P", "e7", "e5")
game += serialize_move("N", "g1", "f3")
game += serialize_move("N", "b8", "c6")

print("token count:", len(game))
print(" ".join(game))

## 2. Next-token prediction 데이터 만들기

사전학습은 prefix를 보고 다음 토큰을 맞히는 학습입니다. 체스 말뭉치에서는 합법적이고 사람다운 수의 분포를 배웁니다.

In [ ]:
def next_token_examples(tokens):
    examples = []
    for i in range(1, len(tokens)):
        examples.append((tokens[:i], tokens[i]))
    return examples


examples = next_token_examples(game)
for prefix, target in examples[:5]:
    print("prefix=", " ".join(prefix), "=> target=", target)

## 3. 합성 reasoning trace 직렬화

논문의 SFT는 가능한 continuation들을 트리처럼 직렬화한 뒤 정답 continuation에 commit하도록 학습합니다. 아래는 그 구조만 간단히 재현한 예입니다.

In [ ]:
candidate_lines = [
    ["<Q>", "<d1>", "<h5>", "<+>"],
    ["<B>", "<f1>", "<c4>", "<->"],
    ["<N>", "<f3>", "<g5>", "<+>"],
]
best_line = candidate_lines[2]


def serialize_reasoning_trace(lines):
    parts = ["<T>"]
    for index, line in enumerate(lines):
        if index:
            parts.append("<SEP>")
        parts.extend(line)
    parts.append("</T>")
    return parts


sft_sequence = serialize_reasoning_trace(candidate_lines) + best_line
print(" ".join(sft_sequence))

## 4. Compute 추정식 이해하기

논문은 dense Transformer 학습 compute를 대략 `6 * N * T` FLOPs로 추정합니다. 여기서 `N`은 파라미터 수, `T`는 처리한 토큰 수입니다.

In [ ]:
def pretraining_flops(parameters, tokens):
    return 6 * parameters * tokens


def human_readable_flops(value):
    units = [(1e18, "EFLOPs"), (1e15, "PFLOPs"), (1e12, "TFLOPs")]
    for scale, unit in units:
        if value >= scale:
            return f"{value / scale:.2f} {unit}"
    return f"{value:.0f} FLOPs"


for parameters, tokens in [(20_000_000, 1_000_000_000), (1_000_000_000, 52_000_000_000)]:
    flops = pretraining_flops(parameters, tokens)
    print(f"N={parameters:,}, T={tokens:,} -> {human_readable_flops(flops)}")